In [74]:
import numpy as np

# Configuration parameters for the (3,4)-regular LDPC base matrix
size = 3     # Example Submatrix size (Z value for WiMAX/WiFi sizes vary, 38 is typical for some max shift values here like 37)
M_b = 3       # Number of macro rows in base matrix
N_b = 4      # Number of macro columns in base matrix

macro_rows= M_b  # Number of layers for layered decoding (often equal to N_b for QC-LDPC)
macro_cols= N_b  # Number of columns in the base matrix (must be >= macro_rows for layered decoding)
n_layers = macro_rows  # Number of bits to print per row (must be >= macro

# Shift values for the quasi-cyclic matrix (-1 means Bypass/All-Zero submatrix)
# Edited to ensure exactly 1 bypass per macro row and macro column.
# shift_matrix = np.array([
#     [ 29, 30,  0,  8, 33, 22, 17,  4, 27, 28, 20, 27, 24, 23, -1,  0],
#     [ 37, 31, 18, 23, 11, 21,  6, 20, 32,  9, 12, 29, 10,  0, 13, -1],
#     [ 25, 22,  4, 34, 31,  3, 14, 15,  4,  2, 14, 18, 13, -1, 22, 24]
# ])

shift_matrix = np.array([
    [ 1, 0,-1, 2],
    [-1, 2, 0, 0],
    [ 1, 0, 2, 1]
])  # Number of macro columns in base matrix

# Shift values for the quasi-cyclic matrix (-1 means Bypass/All-Zero submatrix)
# Edited to ensure exactly 1 bypass per macro row and macro column.
# shift_matrix = np.array([
#     [ 29, 30,  0,  8, 33, 22, 17,  4, 27, 28, 20, 27, 24, 23, -1,  0],
#     [ 37, 31, 18, 23, 11, 21,  6, 20, 32,  9, 12, 29, 10,  0, 13, -1],
#     [ 25, 22,  4, 34, 31,  3, 14, 15,  4,  2, 14, 18, 13, -1, 22, 24]
# ])

M = M_b * size
N = N_b * size

# Generate the full parity-check matrix H
matrix_orig = np.zeros((M, N), dtype=int)

for i in range(M_b):
    for j in range(N_b):
        shift = shift_matrix[i, j]
        row_start = i * size
        col_start = j * size
        
        if shift == -1:
            submat = np.zeros((size, size), dtype=int)
        else:
            submat = np.roll(np.eye(size, dtype=int), shift, axis=1)
            
        matrix_orig[row_start:row_start+size, col_start:col_start+size] = submat

matrix = matrix_orig.copy()
print("Generated Full H Matrix:")
print(matrix_orig.shape)



Generated Full H Matrix:
(9, 12)


In [75]:
# Convert per-row one-hot vectors to single multi-bit integer per row (bypass_tbl).
import numpy as np

def one_hot_bypass_rows_to_int(matrix: np.ndarray, block_size: int, n_layers: int):
    """
    Returns:
      bypass_tbl: list[int]     -- length = number of full rows in matrix
      block_zero: np.ndarray   -- boolean map of macro-rows x macro-columns
    Note: n_layers is accepted so callers can choose how many bits to consider/print.
    """
    rows, cols = matrix.shape
    assert rows % block_size == 0 and cols % block_size == 0, "matrix dims must be multiples of block_size"
    R = rows // block_size
    C = cols // block_size
    
    # detect zero blocks
    block_zero = np.zeros((R, C), dtype=bool)
    for i in range(R):
        for j in range(C):
            block = matrix[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size]
            block_zero[i, j] = not np.any(block)
    
    # build integer bitmask per full row
    bypass_tbl = []
    for r in range(rows):
        mrow = r // block_size
        mask = 0
        for j in range(C):
            if block_zero[mrow, j]:
                mask |= (1 << j)   # set bit j (LSB == macro-column 0)
        bypass_tbl.append(mask)
    return bypass_tbl, block_zero

# --- Demo: create the same toy block matrix used earlier ---
def rot_eye(s, shift):
    return np.roll(np.eye(s, dtype=int), shift, axis=1)

blocks = []
for i in range(macro_rows):
    row_blocks = []
    for j in range(macro_cols):
        if j == (i+1) % macro_cols:  # pick one bypass position per macro-row
            row_blocks.append(np.zeros((size, size), dtype=int))
        else:
            shift = (i*macro_cols + j) % size
            row_blocks.append(rot_eye(size, shift))
    blocks.append(row_blocks)

full = np.block([[blocks[i][j] for j in range(macro_cols)] for i in range(macro_rows)])

bypass_tbl, block_zero = one_hot_bypass_rows_to_int(full, size, n_layers)

mask_trunc = (1 << n_layers) - 1  # mask to keep only the lower n_layers bits
for r, mask in enumerate(bypass_tbl):
    # print exactly n_layers bits (LSB = rightmost bit). If n_layers < macro_cols this truncates higher bits.
    print(f"    bypass_tbl[{r}] = {n_layers}'b{(mask & mask_trunc):0{n_layers}b};")

    bypass_tbl[0] = 3'b010;
    bypass_tbl[1] = 3'b010;
    bypass_tbl[2] = 3'b010;
    bypass_tbl[3] = 3'b100;
    bypass_tbl[4] = 3'b100;
    bypass_tbl[5] = 3'b100;
    bypass_tbl[6] = 3'b000;
    bypass_tbl[7] = 3'b000;
    bypass_tbl[8] = 3'b000;


In [76]:
# Generate cn_pe_tbl and cn_ly_tbl for gen.v
# cn_pe_tbl[c*MAX_DC+d] = PE index of d-th member of CN c
# cn_ly_tbl[c*MAX_DC+d] = layer index of that member
# Sentinel: PE index = N for unused slots (degree < MAX_DC)

rows, cols = matrix_orig.shape
n_layers = rows // size
MAX_DC = max(np.sum(matrix_orig, axis=1))  # max CN degree

# print(f"        // ----- CN member table (cn_pe_tbl / cn_ly_tbl) -----")
# print(f"        // MAX_DC = {MAX_DC}")
for r in range(rows):
    ones_idx = np.where(matrix_orig[r, :] == 1)[0].tolist()
    layer = r // size
    degree = len(ones_idx)

    # Comment line
    members = ", ".join(f"PE{pe}(L{layer})" for pe in ones_idx)
    # print(f"        // CN{r}: {members}")

    # Pad with sentinel to MAX_DC
    padded_pe = ones_idx + ["N"] * (MAX_DC - degree)
    padded_ly = [layer] * degree + [0] * (MAX_DC - degree)

    # Print two entries per line
    base = r * MAX_DC
    for i in range(0, MAX_DC, 2):
        parts = []
        for j in range(i, min(i + 2, MAX_DC)):
            idx = base + j
            pe_str = str(padded_pe[j]) if isinstance(padded_pe[j], int) else padded_pe[j]
            parts.append(f"cn_pe_tbl[{idx}]={pe_str:<2}; cn_ly_tbl[{idx}]={padded_ly[j]}")
        print("        " + ";  ".join(parts) + ";")


        cn_pe_tbl[0]=1 ; cn_ly_tbl[0]=0;  cn_pe_tbl[1]=3 ; cn_ly_tbl[1]=0;
        cn_pe_tbl[2]=11; cn_ly_tbl[2]=0;  cn_pe_tbl[3]=N ; cn_ly_tbl[3]=0;
        cn_pe_tbl[4]=2 ; cn_ly_tbl[4]=0;  cn_pe_tbl[5]=4 ; cn_ly_tbl[5]=0;
        cn_pe_tbl[6]=9 ; cn_ly_tbl[6]=0;  cn_pe_tbl[7]=N ; cn_ly_tbl[7]=0;
        cn_pe_tbl[8]=0 ; cn_ly_tbl[8]=0;  cn_pe_tbl[9]=5 ; cn_ly_tbl[9]=0;
        cn_pe_tbl[10]=10; cn_ly_tbl[10]=0;  cn_pe_tbl[11]=N ; cn_ly_tbl[11]=0;
        cn_pe_tbl[12]=5 ; cn_ly_tbl[12]=1;  cn_pe_tbl[13]=6 ; cn_ly_tbl[13]=1;
        cn_pe_tbl[14]=9 ; cn_ly_tbl[14]=1;  cn_pe_tbl[15]=N ; cn_ly_tbl[15]=0;
        cn_pe_tbl[16]=3 ; cn_ly_tbl[16]=1;  cn_pe_tbl[17]=7 ; cn_ly_tbl[17]=1;
        cn_pe_tbl[18]=10; cn_ly_tbl[18]=1;  cn_pe_tbl[19]=N ; cn_ly_tbl[19]=0;
        cn_pe_tbl[20]=4 ; cn_ly_tbl[20]=1;  cn_pe_tbl[21]=8 ; cn_ly_tbl[21]=1;
        cn_pe_tbl[22]=11; cn_ly_tbl[22]=1;  cn_pe_tbl[23]=N ; cn_ly_tbl[23]=0;
        cn_pe_tbl[24]=1 ; cn_ly_tbl[24]=2;  cn_pe_tbl[25]=3 ; cn_ly_tbl[

In [77]:
# Generate mcv_cn_tbl: PE layer -> CN summary routing
# mcv_cn_tbl[p*n_layers + l] = CN index for PE p, layer l
# Sentinel M (= rows) means bypassed layer

rows, cols = matrix_orig.shape
n_layers = rows // size

# print("        // ----- MCV routing table -----")
for p in range(cols):
    parts = []
    for l in range(n_layers):
        cn = None
        for r in range(l * size, (l + 1) * size):
            if matrix_orig[r, p] == 1:
                cn = r
                break
        parts.append((l, cn))

    comment = ", ".join(
        f"L{l}=CN{cn}" if cn is not None else f"L{l}=bypass"
        for l, cn in parts
    )
    # print(f"        // PE{p}:  {comment}")

    entries = []
    for l, cn in parts:
        idx = p * n_layers + l
        if cn is not None:
            entries.append(f"mcv_cn_tbl[{idx}]={cn}")
        else:
            entries.append(f"mcv_cn_tbl[{idx}]=M")
    print("        " + ";  ".join(entries) + ";")


        mcv_cn_tbl[0]=2;  mcv_cn_tbl[1]=M;  mcv_cn_tbl[2]=8;
        mcv_cn_tbl[3]=0;  mcv_cn_tbl[4]=M;  mcv_cn_tbl[5]=6;
        mcv_cn_tbl[6]=1;  mcv_cn_tbl[7]=M;  mcv_cn_tbl[8]=7;
        mcv_cn_tbl[9]=0;  mcv_cn_tbl[10]=4;  mcv_cn_tbl[11]=6;
        mcv_cn_tbl[12]=1;  mcv_cn_tbl[13]=5;  mcv_cn_tbl[14]=7;
        mcv_cn_tbl[15]=2;  mcv_cn_tbl[16]=3;  mcv_cn_tbl[17]=8;
        mcv_cn_tbl[18]=M;  mcv_cn_tbl[19]=3;  mcv_cn_tbl[20]=7;
        mcv_cn_tbl[21]=M;  mcv_cn_tbl[22]=4;  mcv_cn_tbl[23]=8;
        mcv_cn_tbl[24]=M;  mcv_cn_tbl[25]=5;  mcv_cn_tbl[26]=6;
        mcv_cn_tbl[27]=1;  mcv_cn_tbl[28]=3;  mcv_cn_tbl[29]=8;
        mcv_cn_tbl[30]=2;  mcv_cn_tbl[31]=4;  mcv_cn_tbl[32]=6;
        mcv_cn_tbl[33]=0;  mcv_cn_tbl[34]=5;  mcv_cn_tbl[35]=7;
